In [1]:
print("Hola")

Hola


In [ ]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from srcDatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments


ModuleNotFoundError: No module named 'src'

In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [4]:
rw_data = dg.generate_RandomWalk(random_params=True, num_passos=200)
rw_data.to_pickle('rw_200p_10000t_rp.pkl')

In [5]:
tb_data = dg.generate_BrownianMotion(random_params=True, num_passos=200)
tb_data.to_pickle('tb_200p_10000t_rp.pkl')

In [6]:
ctrw_data = dg.generate_ContinuousTimeRandomWalk(random_params=True, num_passos=200)
ctrw_data.to_pickle('ctrw_200p_10000t_rp.pkl')

In [7]:
fbm_data = dg.generate_FractionalBrownianMotion(random_params=True, num_passos=200)
fbm_data.to_pickle('fbm_200p_10000t_rp.pkl')

c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\fbm\fbm.py:171: UserWarning: Combination of increments n and Hurst value H invalid for Davies-Harte method. Reverting to Hosking method. Occurs when n is small and Hurst is close to 1. 
  warnings.warn(


In [8]:
lw_data = dg.generate_LevyWalks(random_params=True, num_passos=200)
lw_data.to_pickle('lw_200p_10000t_rp.pkl')

In [9]:
attm_data = dg.generate_AnnealedTransitTimeModel(random_params=True, num_passos=200)
attm_data.to_pickle('attm_200p_10000t_rp.pkl')

In [10]:
sbm_data = dg.generate_ScaledBrownianMotion(random_params=True, num_passos=200)
sbm_data.to_pickle('sbm_200p_10000t_rp.pkl')

In [11]:
import pandas as pd
import pickle

with open('./attm_200p_10000t_rp.pkl', 'rb') as file:
    attm_data = pickle.load(file)
print("attm_data completed")
with open('ctrw_200p_10000t_rp.pkl', 'rb') as file:
    ctrw_data = pickle.load(file)
print("ctrw_data completed")
with open('fbm_200p_10000t_rp.pkl', 'rb') as file:
    fbm_data = pickle.load(file)
print("fbm_data completed")
with open('lw_200p_10000t_rp.pkl', 'rb') as file:
    lw_data = pickle.load(file)
print("lw_data completed")
with open('rw_200p_10000t_rp.pkl', 'rb') as file:
    rw_data = pickle.load(file)
print("rw_data completed")
with open('sbm_200p_10000t_rp.pkl', 'rb') as file:
    sbm_data = pickle.load(file)
print("sbm_data completed")
with open('tb_200p_10000t_rp.pkl', 'rb') as file:
    tb_data = pickle.load(file)
print("tb_data completed")

data = pd.concat([attm_data, ctrw_data, fbm_data, lw_data, rw_data, sbm_data, tb_data], ignore_index=True)
data = data.drop(columns=['num_steps', 'attributes'])
data.to_pickle('all_types_200p_70000t_rp.pkl')

attm_data completed
ctrw_data completed
fbm_data completed
lw_data completed
rw_data completed
sbm_data completed
tb_data completed


In [12]:
import pickle
with open('all_types_200p_70000t_rp.pkl', 'rb') as file:
    data = pickle.load(file)

data = data.sample(frac=1).reset_index(drop=True)
data

,coords,trajectory_type
0,"[[2.252711200397771, 2.238749026687866, 2.2030...",CTRW
1,"[[0.0, 4.349567150477632, -1.0020687164967086,...",Brownian Motion
2,"[[2.245195361195648, 1.7453859860776804, 2.800...",Scaled Brownian Motion
3,"[[0.0, 2.5875534460561966, 3.9711958819994546,...",Brownian Motion
4,"[[-0.7209998722501522, 0.08994705947516224, -0...",ATTM
...,...,...
69995,"[[0.0, 0.023395162291921137, 0.031739991597381...",Fractional Brownian Motion
69996,"[[2.5932945153200793, -0.35360802360857724, 1....",ATTM
69997,"[[0.0, 7.0844269093238985, 10.241034022264394,...",Brownian Motion
69998,"[[0.3087868920472065, 0.9030740775332993, 2.33...",CTRW


In [13]:
print(data.columns)

Index(['coords', 'trajectory_type'], dtype='object')


In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [15]:
valores_unicos = data['trajectory_type'].unique()
print(valores_unicos)

['CTRW' 'Brownian Motion' 'Scaled Brownian Motion' 'ATTM' 'Lévy Walk'
 'Fractional Brownian Motion' 'Random Walk']


In [16]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in data['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [17]:
# 2. Convertir las etiquetas a numéricas
le = LabelEncoder()
labels = le.fit_transform(data['trajectory_type'])
labels_tensor  = torch.tensor(labels, dtype=torch.long)
y_tensor = F.one_hot(labels_tensor, num_classes=7).float()

In [18]:
# 3. Reorganizar dimensiones para Conv1D (n_samples, channels, seq_length)
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [19]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [20]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [21]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c
    
# Antic
# class TrajectoryCNN(nn.Module):
#     def __init__(self, input_channels, seq_length, num_classes):
#         super().__init__()
#         self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
#         self.pool = nn.MaxPool1d(2)
        
#         # Cálculo dinámico de la dimensión final
#         self.fc_input_size = self._get_fc_size(seq_length, input_channels)
        
#         self.fc1 = nn.Linear(self.fc_input_size, 128)
#         self.fc2 = nn.Linear(128, num_classes)
    
#     def _get_fc_size(self, seq_length, channels):
#         dummy = torch.zeros(1, channels, seq_length)
#         dummy = self.pool(F.relu(self.conv1(dummy)))
#         dummy = self.pool(F.relu(self.conv2(dummy)))
#         return dummy.view(1, -1).size(1)
    
#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = x.view(x.size(0), -1)
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x


    # def forward(self, x):
    #     print(f"Input shape: {x.shape}")  # (batch_size, channels, seq_length)
        
    #     x = self.pool(F.relu(self.conv1(x)))
    #     print(f"After conv1 + relu + pool: {x.shape}")
        
    #     x = self.pool(F.relu(self.conv2(x)))
    #     print(f"After conv2 + relu + pool: {x.shape}")
        
    #     x = x.view(x.size(0), -1)
    #     print(f"After flatten: {x.shape}")
        
    #     x = F.relu(self.fc1(x))
    #     print(f"After fc1 + relu: {x.shape}")
        
    #     x = self.fc2(x)
    #     print(f"Output shape (logits): {x.shape}")
    
        # return x


In [22]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)

In [23]:
model.first_conv.weight.shape

torch.Size([32, 2, 3])

In [24]:
# 8. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 1.2574
Epoch [2/10], Loss: 0.6911
Epoch [3/10], Loss: 0.5994
Epoch [4/10], Loss: 0.5660
Epoch [5/10], Loss: 0.5397
Epoch [6/10], Loss: 0.5250
Epoch [7/10], Loss: 0.5163
Epoch [8/10], Loss: 0.5078
Epoch [9/10], Loss: 0.4991
Epoch [10/10], Loss: 0.4684


In [26]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Índices de las clases predichas
    true_labels = torch.argmax(y_test, dim=1)  # Convertir one-hot a índices
    accuracy = (predicted == true_labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.8461


In [ ]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'trajectory_cnn_model_2.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'trajectory_cnn_model_complete_2.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()